# Pretrained Model Fine-Tuning for AI-Generated Image Detection

This notebook builds the **modeling stage** of the project and explicitly uses the earlier profiling notebook as guidance:

- Profiling reference: `notebooks/01_ai_generated_image_detection.ipynb`
- Current goal: train a **high-F1** binary classifier for `real` vs `AI-generated`

We compare three families of approaches:

1. **Artifact-based model**
   A pretrained CNN fine-tuned on RGB images.
2. **Spectrum-based model**
   A dual-branch model that combines RGB images and FFT-spectrum images.
3. **Image-encoder-based model**
   A pretrained ViT-style encoder with a classifier head.

The notebook is optimized around **F1 score**, not just accuracy.


## 1. Project Setup

We load the local image dataset, import the reusable training helpers from `src/ai_image_detection/pretrained.py`, and verify the expected data paths.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'Data'
IMAGE_DIR = DATA_DIR / 'images_final_sample'
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

required_paths = [TRAIN_CSV, TEST_CSV, IMAGE_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('Missing expected dataset paths: ' + ', '.join(missing_paths))

print('Project root:', PROJECT_ROOT)
print('Data directory:', DATA_DIR)
print('Artifacts directory:', ARTIFACTS_DIR)

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import ConfusionMatrixDisplay
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

from ai_image_detection.data import create_train_val_split, load_metadata
from ai_image_detection.engine import seed_everything, select_device
from ai_image_detection.pretrained import (
    FineTuneImageDataset,
    build_artifact_model,
    build_encoder_model,
    build_eval_transforms,
    build_spectrum_model,
    build_submission,
    build_train_transforms,
    evaluate_predictions,
    find_best_f1_threshold,
    freeze_backbone,
    predict_pretrained_probabilities,
    train_binary_model,
    unfreeze_last_layers,
)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
seed_everything(42)
device = select_device()
print('Using device:', device)

## 2. Load Data and Revisit Profiling Insights

We reuse the same metadata-loading approach as the profiling notebook. Before training, we restate the most important insight from profiling:

- AI-vs-real separation may come from texture, frequency artifacts, metadata presence, and compression behavior
- so we should not rely on a single model family
- we want both deep features and profiling-inspired signals


In [ ]:
train_df = load_metadata(TRAIN_CSV, IMAGE_DIR)
test_df = load_metadata(TEST_CSV, IMAGE_DIR)

train_df = train_df.loc[train_df['is_available']].copy().reset_index(drop=True)
test_df = test_df.loc[test_df['is_available']].copy().reset_index(drop=True)
train_df['label_name'] = train_df['ground_truth'].map({0: 'real', 1: 'ai_generated'})

print(f'Train images available: {len(train_df):,}')
print(f'Test images available: {len(test_df):,}')
class_dist = train_df['label_name'].value_counts().rename_axis('label').to_frame('count')
class_dist['share'] = class_dist['count'] / class_dist['count'].sum()
display(class_dist)

In [ ]:
train_split_df, val_split_df = create_train_val_split(train_df, val_size=0.2, random_state=42)

split_summary = pd.DataFrame(
    {
        'rows': [len(train_split_df), len(val_split_df)],
        'real': [int((train_split_df['ground_truth'] == 0).sum()), int((val_split_df['ground_truth'] == 0).sum())],
        'ai_generated': [int((train_split_df['ground_truth'] == 1).sum()), int((val_split_df['ground_truth'] == 1).sum())],
    },
    index=['train', 'validation'],
)
print('Train/validation split summary:')
display(split_summary)

## 3. Data Pipeline and Augmentations

We use ImageNet normalization and pretrained-model-friendly transforms.

Training only:
- resize + crop
- horizontal flip
- color jitter

Validation/test:
- deterministic resize
- normalization only


In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0

train_transform = build_train_transforms(image_size=IMAGE_SIZE)
eval_transform = build_eval_transforms(image_size=IMAGE_SIZE)

transform_summary = pd.DataFrame(
    {
        'split': ['train', 'validation/test'],
        'transform_pipeline': [
            'Resize -> RandomCrop -> RandomHorizontalFlip -> ColorJitter -> Normalize',
            'Resize -> Normalize',
        ],
    }
)
print('Transform setup:')
display(transform_summary)

In [ ]:
train_dataset = FineTuneImageDataset(train_split_df, image_transform=train_transform)
val_dataset = FineTuneImageDataset(val_split_df, image_transform=eval_transform)
test_dataset = FineTuneImageDataset(test_df, image_transform=eval_transform, target_column=None)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

first_batch = next(iter(train_loader))
print('Train batch keys:', list(first_batch.keys()))
print('Image batch shape:', tuple(first_batch['image'].shape))
print('Target batch shape:', tuple(first_batch['target'].shape))

## 4. F1-Driven Training Strategy

This project optimizes for **F1 score**.

That means:
- we use `BCEWithLogitsLoss`
- we compute validation probabilities after every epoch
- we search thresholds in `[0.3, 0.7]`
- we pick the threshold that maximizes validation F1 rather than using `0.5` blindly


In [ ]:
train_strategy_df = pd.DataFrame(
    {
        'component': ['loss', 'optimizer', 'scheduler', 'primary_metric', 'threshold_search'],
        'choice': ['BCEWithLogitsLoss', 'AdamW', 'CosineAnnealingLR', 'F1 score', '[0.3, 0.7]'],
    }
)
print('Training strategy:')
display(train_strategy_df)

## 5. Model 1: Artifact-Based Fine-Tuned CNN

This is the strongest and simplest baseline.

It uses a pretrained CNN to learn:
- local texture inconsistencies
- compression artifacts
- visual generation artifacts

We start with **EfficientNet-B0**, first training the head and then optionally unfreezing more layers.


In [ ]:
artifact_bundle = build_artifact_model(model_name='efficientnet_b0', pretrained=True)
artifact_model = artifact_bundle.model.to(device)
freeze_backbone(artifact_model)

criterion = nn.BCEWithLogitsLoss()
artifact_optimizer = AdamW(filter(lambda p: p.requires_grad, artifact_model.parameters()), lr=1e-3, weight_decay=1e-4)
artifact_scheduler = CosineAnnealingLR(artifact_optimizer, T_max=3)

print('Artifact model:', artifact_bundle.model_name)
print('Input size:', artifact_bundle.input_size)
print('Backbone frozen; classifier head trainable.')

In [ ]:
artifact_history_phase1 = train_binary_model(
    artifact_model,
    train_loader,
    val_loader,
    device=device,
    epochs=3,
    optimizer=artifact_optimizer,
    scheduler=artifact_scheduler,
    loss_fn=criterion,
)
print('Artifact model phase-1 history:')
display(artifact_history_phase1)

In [ ]:
artifact_val_probs = predict_pretrained_probabilities(artifact_model, val_loader, device=device)
artifact_val_targets = val_split_df['ground_truth'].to_numpy()
artifact_threshold, artifact_best_f1 = find_best_f1_threshold(artifact_val_targets, artifact_val_probs)
artifact_metrics = evaluate_predictions(artifact_val_targets, artifact_val_probs, artifact_threshold)

artifact_metrics_table = pd.DataFrame(
    {
        'metric': ['f1', 'precision', 'recall', 'accuracy', 'best_threshold'],
        'value': [
            artifact_metrics['f1'],
            artifact_metrics['precision'],
            artifact_metrics['recall'],
            artifact_metrics['accuracy'],
            artifact_threshold,
        ],
    }
)
print('Artifact model validation metrics:')
display(artifact_metrics_table)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(artifact_metrics['confusion_matrix'], display_labels=['real', 'ai_generated']).plot(ax=ax, colorbar=False)
ax.set_title('Artifact Model Confusion Matrix')
plt.tight_layout()

## 6. Model 2: Spectrum-Based Model

This model uses both:
- RGB image input
- FFT-spectrum image input

Why?
Because profiling suggested that frequency artifacts can help distinguish AI-generated images from real ones.


In [ ]:
spectrum_train_dataset = FineTuneImageDataset(
    train_split_df,
    image_transform=train_transform,
    include_fft_image=True,
)
spectrum_val_dataset = FineTuneImageDataset(
    val_split_df,
    image_transform=eval_transform,
    include_fft_image=True,
)
spectrum_test_dataset = FineTuneImageDataset(
    test_df,
    image_transform=eval_transform,
    target_column=None,
    include_fft_image=True,
)

spectrum_train_loader = DataLoader(spectrum_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
spectrum_val_loader = DataLoader(spectrum_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
spectrum_test_loader = DataLoader(spectrum_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

spectrum_batch = next(iter(spectrum_train_loader))
print('Spectrum batch keys:', list(spectrum_batch.keys()))
print('RGB image shape:', tuple(spectrum_batch['image'].shape))
print('FFT image shape:', tuple(spectrum_batch['fft_image'].shape))

In [ ]:
spectrum_bundle = build_spectrum_model(model_name='efficientnet_b0', pretrained=True)
spectrum_model = spectrum_bundle.model.to(device)

spectrum_optimizer = AdamW(spectrum_model.parameters(), lr=1e-4, weight_decay=1e-4)
spectrum_scheduler = CosineAnnealingLR(spectrum_optimizer, T_max=3)

spectrum_history = train_binary_model(
    spectrum_model,
    spectrum_train_loader,
    spectrum_val_loader,
    device=device,
    epochs=3,
    optimizer=spectrum_optimizer,
    scheduler=spectrum_scheduler,
    loss_fn=criterion,
)
print('Spectrum model history:')
display(spectrum_history)

In [ ]:
spectrum_val_probs = predict_pretrained_probabilities(spectrum_model, spectrum_val_loader, device=device)
spectrum_threshold, _ = find_best_f1_threshold(artifact_val_targets, spectrum_val_probs)
spectrum_metrics = evaluate_predictions(artifact_val_targets, spectrum_val_probs, spectrum_threshold)

spectrum_metrics_table = pd.DataFrame(
    {
        'metric': ['f1', 'precision', 'recall', 'accuracy', 'best_threshold'],
        'value': [
            spectrum_metrics['f1'],
            spectrum_metrics['precision'],
            spectrum_metrics['recall'],
            spectrum_metrics['accuracy'],
            spectrum_threshold,
        ],
    }
)
print('Spectrum model validation metrics:')
display(spectrum_metrics_table)

## 7. Model 3: Image-Encoder-Based Model

This is the high-level semantic model.

We use a pretrained **Vision Transformer** and fine-tune a classifier head. The idea is that encoder-style models may generalize better to unseen generation styles.


In [ ]:
encoder_bundle = build_encoder_model(model_name='vit_b_16', pretrained=True)
encoder_model = encoder_bundle.model.to(device)
freeze_backbone(encoder_model)

encoder_optimizer = AdamW(filter(lambda p: p.requires_grad, encoder_model.parameters()), lr=1e-3, weight_decay=1e-4)
encoder_scheduler = CosineAnnealingLR(encoder_optimizer, T_max=3)

encoder_history = train_binary_model(
    encoder_model,
    train_loader,
    val_loader,
    device=device,
    epochs=3,
    optimizer=encoder_optimizer,
    scheduler=encoder_scheduler,
    loss_fn=criterion,
)
print('Encoder model history:')
display(encoder_history)

In [ ]:
encoder_val_probs = predict_pretrained_probabilities(encoder_model, val_loader, device=device)
encoder_threshold, _ = find_best_f1_threshold(artifact_val_targets, encoder_val_probs)
encoder_metrics = evaluate_predictions(artifact_val_targets, encoder_val_probs, encoder_threshold)

encoder_metrics_table = pd.DataFrame(
    {
        'metric': ['f1', 'precision', 'recall', 'accuracy', 'best_threshold'],
        'value': [
            encoder_metrics['f1'],
            encoder_metrics['precision'],
            encoder_metrics['recall'],
            encoder_metrics['accuracy'],
            encoder_threshold,
        ],
    }
)
print('Encoder model validation metrics:')
display(encoder_metrics_table)

## 8. Robustness Checks Under Perturbations

We explicitly evaluate distribution shift under:
- JPEG compression
- Gaussian blur

This matters because a detector that only works on the clean validation split may fail in real-world settings.


In [ ]:
def evaluate_with_perturbation(model, records, threshold, perturbation_name):
    dataset = FineTuneImageDataset(
        records,
        image_transform=eval_transform,
        perturbation=perturbation_name,
    )
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    probs = predict_pretrained_probabilities(model, loader, device=device)
    return evaluate_predictions(records['ground_truth'].to_numpy(), probs, threshold)

artifact_jpeg_metrics = evaluate_with_perturbation(artifact_model, val_split_df, artifact_threshold, 'jpeg')
artifact_blur_metrics = evaluate_with_perturbation(artifact_model, val_split_df, artifact_threshold, 'blur')

robustness_df = pd.DataFrame(
    [
        {'scenario': 'clean', 'f1': artifact_metrics['f1'], 'precision': artifact_metrics['precision'], 'recall': artifact_metrics['recall'], 'accuracy': artifact_metrics['accuracy']},
        {'scenario': 'jpeg', 'f1': artifact_jpeg_metrics['f1'], 'precision': artifact_jpeg_metrics['precision'], 'recall': artifact_jpeg_metrics['recall'], 'accuracy': artifact_jpeg_metrics['accuracy']},
        {'scenario': 'blur', 'f1': artifact_blur_metrics['f1'], 'precision': artifact_blur_metrics['precision'], 'recall': artifact_blur_metrics['recall'], 'accuracy': artifact_blur_metrics['accuracy']},
    ]
)
robustness_df['f1_drop_vs_clean'] = artifact_metrics['f1'] - robustness_df['f1']
print('Artifact model robustness table:')
display(robustness_df)

## 9. Optional Fusion of Handcrafted Features and Deep Features

Profiling suggested that some handcrafted signals may matter:
- Laplacian variance
- FFT high-frequency ratio
- RGB statistics
- EXIF presence
- file size

This cell sets up a fusion-ready dataset. You can use it to test whether handcrafted features improve the deep model.


In [ ]:
fusion_train_dataset = FineTuneImageDataset(
    train_split_df,
    image_transform=train_transform,
    include_handcrafted=True,
)
fusion_val_dataset = FineTuneImageDataset(
    val_split_df,
    image_transform=eval_transform,
    include_handcrafted=True,
)

fusion_train_loader = DataLoader(fusion_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
fusion_val_loader = DataLoader(fusion_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

fusion_batch = next(iter(fusion_train_loader))
handcrafted_feature_names = [
    'rgb_mean_r', 'rgb_mean_g', 'rgb_mean_b',
    'rgb_std_r', 'rgb_std_g', 'rgb_std_b',
    'pixel_std', 'laplacian_proxy', 'fft_high_freq_ratio',
    'has_exif', 'file_size_kb',
]
print('Handcrafted feature vector shape:', tuple(fusion_batch['handcrafted'].shape))
display(pd.DataFrame({'feature_name': handcrafted_feature_names}))

## 10. Model Comparison Table

This is the primary comparison table for the three pretrained-model categories.


In [ ]:
comparison_df = pd.DataFrame(
    [
        {'model_family': 'artifact_cnn', 'backbone': artifact_bundle.model_name, 'f1': artifact_metrics['f1'], 'precision': artifact_metrics['precision'], 'recall': artifact_metrics['recall'], 'accuracy': artifact_metrics['accuracy'], 'threshold': artifact_threshold},
        {'model_family': 'spectrum_dual_branch', 'backbone': spectrum_bundle.model_name, 'f1': spectrum_metrics['f1'], 'precision': spectrum_metrics['precision'], 'recall': spectrum_metrics['recall'], 'accuracy': spectrum_metrics['accuracy'], 'threshold': spectrum_threshold},
        {'model_family': 'image_encoder', 'backbone': encoder_bundle.model_name, 'f1': encoder_metrics['f1'], 'precision': encoder_metrics['precision'], 'recall': encoder_metrics['recall'], 'accuracy': encoder_metrics['accuracy'], 'threshold': encoder_threshold},
    ]
).sort_values('f1', ascending=False)
print('Model comparison table (sorted by validation F1):')
display(comparison_df)

## 11. Failure Analysis

We inspect the artifact-model validation predictions and look at examples where:
- real images were misclassified as AI
- AI-generated images were missed as real

This is where model diagnostics become actionable.


In [ ]:
val_analysis_df = val_split_df[['image_id', 'image_path', 'ground_truth']].copy()
val_analysis_df['prob_ai'] = artifact_val_probs
val_analysis_df['pred_label'] = (artifact_val_probs >= artifact_threshold).astype(int)
val_analysis_df['error_type'] = 'correct'
val_analysis_df.loc[(val_analysis_df['ground_truth'] == 0) & (val_analysis_df['pred_label'] == 1), 'error_type'] = 'false_positive_real_as_ai'
val_analysis_df.loc[(val_analysis_df['ground_truth'] == 1) & (val_analysis_df['pred_label'] == 0), 'error_type'] = 'false_negative_ai_as_real'

error_summary = val_analysis_df['error_type'].value_counts().to_frame('count')
print('Validation prediction outcome summary:')
display(error_summary)

display(val_analysis_df.head())

In [ ]:
false_positive_examples = val_analysis_df.loc[val_analysis_df['error_type'] == 'false_positive_real_as_ai'].head(4)
false_negative_examples = val_analysis_df.loc[val_analysis_df['error_type'] == 'false_negative_ai_as_real'].head(4)
examples = pd.concat([false_positive_examples, false_negative_examples], ignore_index=True)

if examples.empty:
    print('No misclassified examples found in the current validation slice.')
else:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    for ax, (_, row) in zip(axes, examples.iterrows()):
        img = Image.open(row['image_path']).convert('RGB')
        ax.imshow(img)
        ax.set_title(f"true={int(row['ground_truth'])} pred={int(row['pred_label'])}\nprob={row['prob_ai']:.3f}\n{row['error_type']}", fontsize=9)
        ax.axis('off')
    for ax in axes[len(examples):]:
        ax.axis('off')
    plt.tight_layout()

## 12. Feature-Space Visualization with PCA / t-SNE

To understand separation quality, we embed validation examples into a lower-dimensional view.

For a lightweight notebook demonstration, we use the artifact-model prediction score plus handcrafted summary features as a compact visualization space.


In [ ]:
viz_df = val_analysis_df[['prob_ai', 'ground_truth']].copy()

sample_size = min(500, len(val_analysis_df))
viz_sample = val_analysis_df.sample(sample_size, random_state=42).copy()
X = viz_sample[['prob_ai']].to_numpy()

if len(viz_sample) >= 3:
    pca = PCA(n_components=1)
    pca_coords = pca.fit_transform(X)
    viz_plot_df = pd.DataFrame({'pc1': pca_coords[:, 0], 'label': viz_sample['ground_truth'].map({0: 'real', 1: 'ai_generated'})})
    plt.figure(figsize=(8, 4))
    sns.histplot(data=viz_plot_df, x='pc1', hue='label', kde=True, stat='density', common_norm=False)
    plt.title('PCA Projection of Validation Prediction Space')
    plt.xlabel('PC1')
    plt.tight_layout()
else:
    print('Not enough validation examples for PCA visualization.')

## 13. Final Test Predictions

We select the best validation-F1 model and generate the final `submission.csv` in the format:

- `image_id`
- `label`


In [ ]:
best_row = comparison_df.iloc[0]
print('Best validation model:')
display(best_row.to_frame('value'))

if best_row['model_family'] == 'artifact_cnn':
    best_model = artifact_model
    best_loader = test_loader
    best_threshold = artifact_threshold
elif best_row['model_family'] == 'spectrum_dual_branch':
    best_model = spectrum_model
    best_loader = spectrum_test_loader
    best_threshold = spectrum_threshold
else:
    best_model = encoder_model
    best_loader = test_loader
    best_threshold = encoder_threshold

best_test_probs = predict_pretrained_probabilities(best_model, best_loader, device=device)
submission_df = build_submission(test_df, best_test_probs, best_threshold)
submission_path = ARTIFACTS_DIR / 'submission_pretrained_finetuning.csv'
submission_df.to_csv(submission_path, index=False)
print('Saved submission to:', submission_path)
display(submission_df.head())

## 14. Final Reasoning and Recommended Next Steps

What this notebook is designed to answer:

- Which pretrained model family gives the strongest **F1**?
- Do frequency cues improve over a plain RGB artifact detector?
- Does a high-level encoder generalize better than a CNN?
- Do handcrafted signals from profiling help deep models?

Recommended production workflow:

1. Use the profiling notebook to understand data signals.
2. Use this notebook to compare pretrained-model families.
3. Keep the strongest model by **validation F1**, not accuracy.
4. Add robustness checks before trusting the model.
5. Only keep handcrafted features if they improve validation F1 consistently.

If the spectrum or encoder model wins, that is evidence the detector is capturing more than simple compression shortcuts.
If the artifact model wins strongly, the dataset may be dominated by visible low-level generation artifacts.
